In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print("PROJECT 4: MARINE TIDAL TURBINE ANALYTICS")
print("-----------------------------------------")
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Environment setup: PASS")


In [ ]:

from google.colab import drive

drive.mount('/content/drive')

project_dir = Path(
    '/content/drive/MyDrive/Project_4_Tidal_Turbine'
)

project_dir.mkdir(parents=True, exist_ok=True)

print("Project folder:", project_dir)
print("Google Drive setup: PASS")


In [ ]:

import requests
import pandas as pd
from pathlib import Path

project_dir = Path(
    "/content/drive/MyDrive/Project_4_Tidal_Turbine"
)

data_dir = project_dir / "data"
data_dir.mkdir(parents=True, exist_ok=True)

base_url = "https://mhkdr.openei.org/files/738/"

files = [
    "CreaForm_Points_260217.xlsx",
    "LB10_250_2026_02_17_15_17_27_10Hz.xlsx",
    "LB9_225_test2_2026_02_17_14_53_00_10Hz.xlsx"
]

print("PROJECT 4: DATASET ACQUISITION")
print("--------------------------------")

for filename in files:
    destination = data_dir / filename

    try:
        if not destination.exists():
            response = requests.get(
                base_url + filename,
                timeout=90
            )
            response.raise_for_status()
            destination.write_bytes(response.content)

        # Verify that the file is a valid Excel workbook
        workbook = pd.ExcelFile(destination)

        print(f"\nFILE: {filename}")
        print("STATUS: PASS")
        print("SIZE:", round(
            destination.stat().st_size / 1024, 2
        ), "KB")
        print("WORKSHEETS:", workbook.sheet_names)

    except Exception as error:
        print(f"\nFILE: {filename}")
        print("STATUS: FAILED")
        print("ERROR:", str(error)[:200])

print("\nDATASET INVENTORY COMPLETE")


In [ ]:

import pandas as pd
from pathlib import Path

data_dir = Path(
    "/content/drive/MyDrive/Project_4_Tidal_Turbine/data"
)

files = [
    "CreaForm_Points_260217.xlsx",
    "LB10_250_2026_02_17_15_17_27_10Hz.xlsx",
    "LB9_225_test2_2026_02_17_14_53_00_10Hz.xlsx"
]

print("PROJECT 4: DATA STRUCTURE VALIDATION")
print("=" * 55)

for filename in files:
    print(f"\nFILE: {filename}")

    path = data_dir / filename

    with pd.ExcelFile(path) as workbook:
        for sheet in workbook.sheet_names:
            print(f"\nWORKSHEET: {sheet}")

            # Read a small sample first to inspect layout
            sample = pd.read_excel(
                workbook,
                sheet_name=sheet,
                nrows=10,
                header=None
            )

            print("FIRST 5 RAW ROWS:")
            print(sample.head(5).to_string(
                index=False,
                header=False,
                max_cols=12
            ))

            # Read the worksheet using its first row as header
            df = pd.read_excel(
                workbook,
                sheet_name=sheet
            )

            print("\nROWS:", len(df))
            print("COLUMNS:", len(df.columns))
            print("COLUMN NAMES:")
            print(df.columns.tolist())

            print("\nDATA TYPES:")
            print(df.dtypes.value_counts().to_dict())

            print("MISSING VALUES:", int(
                df.isna().sum().sum()
            ))

            print("-" * 55)

print("\nCELL 4 STRUCTURE INSPECTION COMPLETE")


In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path

data_dir = Path(
    "/content/drive/MyDrive/Project_4_Tidal_Turbine/data"
)

test_files = {
    "225% Load Case":
        "LB9_225_test2_2026_02_17_14_53_00_10Hz.xlsx",
    "250% Load Case":
        "LB10_250_2026_02_17_15_17_27_10Hz.xlsx"
}

datasets = {}

print("PROJECT 4: ENGINEERING DATA QUALITY")
print("=" * 55)

for name, filename in test_files.items():

    df = pd.read_excel(
        data_dir / filename,
        sheet_name="SlowData"
    )

    datasets[name] = df

    numeric = df.select_dtypes(include="number")

    missing = int(df.isna().sum().sum())
    duplicates = int(df.duplicated().sum())

    nonfinite = int(
        np.isinf(numeric.to_numpy()).sum()
    )

    constant = [
        col for col in numeric.columns
        if numeric[col].nunique(dropna=True) <= 1
    ]

    time = pd.to_numeric(
        df["LabVIEW Timestamp"],
        errors="coerce"
    )

    intervals = time.diff().dropna()

    invalid_intervals = int(
        (intervals <= 0).sum()
    )

    scan_errors = df["Scan Errors"]

    print(f"\nTEST: {name}")
    print("-" * 55)

    print("Records:", len(df))
    print("Columns:", len(df.columns))
    print("Missing values:", missing)
    print("Duplicate records:", duplicates)
    print("Infinite values:", nonfinite)
    print("Constant channels:", constant)

    print("\nTIMESTAMP VALIDATION")
    print("Duration (seconds):",
          round(time.iloc[-1] - time.iloc[0], 2))
    print("Median sampling interval:",
          round(intervals.median(), 6))
    print("Invalid time intervals:", invalid_intervals)
    print("Intervals outside 0.09–0.11 seconds:",
          int(((intervals < 0.09) |
               (intervals > 0.11)).sum()))

    print("\nACQUISITION INDICATORS")
    print("Scan Errors unique values:",
          scan_errors.unique()[:10])
    print("Late Scans min/max:",
          df["Late Scans"].min(),
          df["Late Scans"].max())
    print("Watchdog unique values:",
          df["Watchdog"].unique()[:10])

    print("\nKEY ENGINEERING CHANNELS")
    for col in [
        "Inboard Load",
        "Outboard Load",
        "Inboard Disp",
        "Outboard Disp",
        "RootMomentUNCORRECTED"
    ]:
        print(
            f"{col}: "
            f"min={df[col].min():.4f}, "
            f"max={df[col].max():.4f}"
        )

print("\nCELL 5 VALIDATION COMPLETE")


In [ ]:

import pandas as pd
from pathlib import Path

data_dir = Path(
    "/content/drive/MyDrive/Project_4_Tidal_Turbine/data"
)

files = {
    "225% Load Case":
        "LB9_225_test2_2026_02_17_14_53_00_10Hz.xlsx",
    "250% Load Case":
        "LB10_250_2026_02_17_15_17_27_10Hz.xlsx"
}

print("PROJECT 4: SENSOR AND METADATA VALIDATION")
print("=" * 60)

for test_name, filename in files.items():

    path = data_dir / filename

    df = pd.read_excel(
        path,
        sheet_name="SlowData"
    )

    print(f"\nTEST: {test_name}")
    print("-" * 60)

    sensor_columns = [
        col for col in df.columns
        if col.startswith("SG")
    ]

    sensor_summary = pd.DataFrame({
        "Sensor": sensor_columns,
        "Unique Values": [
            df[col].nunique()
            for col in sensor_columns
        ],
        "Minimum": [
            df[col].min()
            for col in sensor_columns
        ],
        "Maximum": [
            df[col].max()
            for col in sensor_columns
        ]
    })

    constant_sensors = sensor_summary[
        sensor_summary["Unique Values"] <= 1
    ]

    print("\nCONSTANT STRAIN GAUGES")
    print(constant_sensors.to_string(index=False))

    print("\nSENSOR COUNT:", len(sensor_columns))
    print("CONSTANT SENSOR COUNT:", len(constant_sensors))

    # Inspect the workbook's acquisition metadata.
    with pd.ExcelFile(path) as workbook:
        metadata_sheet = [
            name for name in workbook.sheet_names
            if name != "SlowData"
        ][0]

        metadata = pd.read_excel(
            workbook,
            sheet_name=metadata_sheet,
            header=None
        )

    print("\nMETADATA ROWS CONTAINING CHANNEL TERMS")

    terms = [
        "strain", "load", "disp",
        "moment", "unit", "channel",
        "sg08", "sg21", "sg23", "sg36"
    ]

    matches = metadata.astype(str).apply(
        lambda row: row.str.lower().str.contains(
            "|".join(terms),
            regex=True
        ).any(),
        axis=1
    )

    selected = metadata.loc[matches]

    print(selected.head(25).to_string(
        index=False,
        header=False,
        max_cols=8,
        max_colwidth=35
    ))

print("\nCELL 6 SENSOR INSPECTION COMPLETE")


In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path

project_dir = Path(
    "/content/drive/MyDrive/Project_4_Tidal_Turbine"
)

data_dir = project_dir / "data"
clean_dir = project_dir / "cleaned_data"
clean_dir.mkdir(parents=True, exist_ok=True)

files = {
    "225": "LB9_225_test2_2026_02_17_14_53_00_10Hz.xlsx",
    "250": "LB10_250_2026_02_17_15_17_27_10Hz.xlsx"
}

quality_records = []
summary_records = []

print("PROJECT 4: ENGINEERING DATA PREPARATION")
print("=" * 60)

for test_id, filename in files.items():

    df = pd.read_excel(
        data_dir / filename,
        sheet_name="SlowData"
    )

    df["test_id"] = test_id

    df["elapsed_seconds"] = (
        df["LabVIEW Timestamp"]
        - df["LabVIEW Timestamp"].iloc[0]
    )

    sensor_columns = [
        col for col in df.columns
        if col.startswith("SG")
    ]

    constant_sensors = [
        col for col in sensor_columns
        if df[col].nunique(dropna=True) <= 1
    ]

    valid_sensors = [
        col for col in sensor_columns
        if col not in constant_sensors
    ]

    # Retain all raw engineering measurements.
    # Create a separate sensor-analysis dataset.
    sensor_df = df[
        ["test_id", "elapsed_seconds"]
        + valid_sensors
    ].copy()

    # Save the complete measurement dataset.
    full_path = clean_dir / f"test_{test_id}_complete.csv"
    df.to_csv(full_path, index=False)

    # Save the selected varying strain-gauge channels.
    sensor_path = clean_dir / f"test_{test_id}_sensors.csv"
    sensor_df.to_csv(sensor_path, index=False)

    for sensor in sensor_columns:
        quality_records.append({
            "test_id": test_id,
            "sensor": sensor,
            "unique_values": df[sensor].nunique(),
            "status": (
                "REVIEW_CONSTANT"
                if sensor in constant_sensors
                else "VARIES"
            )
        })

    summary_records.append({
        "test_id": test_id,
        "records": len(df),
        "total_sensors": len(sensor_columns),
        "varying_sensors": len(valid_sensors),
        "constant_sensors": len(constant_sensors)
    })

    print(f"\nTEST: {test_id}%")
    print("Records:", len(df))
    print("Total strain gauges:", len(sensor_columns))
    print("Varying gauges:", len(valid_sensors))
    print("Flagged constant gauges:", constant_sensors)
    print("Complete CSV:", full_path.name)
    print("Sensor CSV:", sensor_path.name)

quality_df = pd.DataFrame(quality_records)
summary_df = pd.DataFrame(summary_records)

quality_df.to_csv(
    clean_dir / "sensor_quality_report.csv",
    index=False
)

summary_df.to_csv(
    clean_dir / "dataset_preparation_summary.csv",
    index=False
)

print("\nDATA PREPARATION SUMMARY")
print(summary_df.to_string(index=False))

print("\nCELL 7 DATA PREPARATION COMPLETE")


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

project_dir = Path(
    "/content/drive/MyDrive/Project_4_Tidal_Turbine"
)

clean_dir = project_dir / "cleaned_data"
dashboard_dir = project_dir / "dashboards"
dashboard_dir.mkdir(parents=True, exist_ok=True)

tests = {
    "225% Load Case": pd.read_csv(
        clean_dir / "test_225_complete.csv"
    ),
    "250% Load Case": pd.read_csv(
        clean_dir / "test_250_complete.csv"
    )
}

print("PROJECT 4: STRUCTURAL PERFORMANCE DASHBOARDS")
print("=" * 60)

# Figure 1: Structural loading
fig, axes = plt.subplots(2, 1, figsize=(13, 9))

for ax, (name, df) in zip(axes, tests.items()):
    ax.plot(
        df["elapsed_seconds"],
        df["Inboard Load"],
        label="Inboard Load"
    )
    ax.plot(
        df["elapsed_seconds"],
        df["Outboard Load"],
        label="Outboard Load"
    )

    ax.set_title(name)
    ax.set_xlabel("Elapsed Time (seconds)")
    ax.set_ylabel("Recorded Load (units unverified)")
    ax.legend()
    ax.grid(alpha=0.3)

fig.suptitle(
    "Tidal Turbine Structural Loading Comparison",
    fontsize=15
)

plt.tight_layout()

load_path = dashboard_dir / "structural_loading.png"
plt.savefig(load_path, dpi=200, bbox_inches="tight")
plt.show()

# Figure 2: Displacement
fig, axes = plt.subplots(2, 1, figsize=(13, 9))

for ax, (name, df) in zip(axes, tests.items()):
    ax.plot(
        df["elapsed_seconds"],
        df["Inboard Disp"],
        label="Inboard Displacement"
    )
    ax.plot(
        df["elapsed_seconds"],
        df["Outboard Disp"],
        label="Outboard Displacement"
    )

    ax.set_title(name)
    ax.set_xlabel("Elapsed Time (seconds)")
    ax.set_ylabel(
        "Recorded Displacement (units unverified)"
    )
    ax.legend()
    ax.grid(alpha=0.3)

fig.suptitle(
    "Tidal Turbine Structural Displacement Comparison",
    fontsize=15
)

plt.tight_layout()

disp_path = dashboard_dir / "structural_displacement.png"
plt.savefig(disp_path, dpi=200, bbox_inches="tight")
plt.show()

print("\nDASHBOARD FILES")
print(load_path)
print(disp_path)

print("\nCELL 8 VISUALIZATION COMPLETE")


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

project_dir = Path(
    "/content/drive/MyDrive/Project_4_Tidal_Turbine"
)

clean_dir = project_dir / "cleaned_data"
dashboard_dir = project_dir / "dashboards"
dashboard_dir.mkdir(parents=True, exist_ok=True)

tests = {
    "225": pd.read_csv(
        clean_dir / "test_225_complete.csv"
    ),
    "250": pd.read_csv(
        clean_dir / "test_250_complete.csv"
    )
}

summary = []

print("PROJECT 4: STRAIN-GAUGE ANALYSIS")
print("=" * 60)

for test_id, df in tests.items():

    sensors = [
        c for c in df.columns
        if c.startswith("SG") and
        df[c].nunique(dropna=True) > 1
    ]

    # Rank by recorded signal range.
    ranges = (
        df[sensors].max() - df[sensors].min()
    ).sort_values(ascending=False)

    top_sensors = ranges.head(6).index.tolist()

    print(f"\nTEST: {test_id}%")
    print("Varying sensors:", len(sensors))
    print("\nTOP 6 SENSOR RANGES")
    print(ranges.head(6).to_string())

    fig, ax = plt.subplots(figsize=(13, 6))

    for sensor in top_sensors:
        ax.plot(
            df["elapsed_seconds"],
            df[sensor],
            label=sensor,
            linewidth=1
        )

    ax.set_title(
        f"{test_id}% Load Case — Strain-Gauge Response"
    )
    ax.set_xlabel("Elapsed Time (seconds)")
    ax.set_ylabel("Recorded Sensor Signal (units unverified)")
    ax.legend(loc="best", fontsize=8)
    ax.grid(alpha=0.3)

    plt.tight_layout()

    output = dashboard_dir / f"strain_response_{test_id}.png"
    plt.savefig(output, dpi=200, bbox_inches="tight")
    plt.show()

    for sensor in sensors:
        summary.append({
            "test_id": test_id,
            "sensor": sensor,
            "minimum": df[sensor].min(),
            "maximum": df[sensor].max(),
            "signal_range": ranges[sensor]
        })

summary_df = pd.DataFrame(summary)

output_csv = clean_dir / "strain_sensor_comparison.csv"
summary_df.to_csv(output_csv, index=False)

print("\nSAVED FILES")
print(dashboard_dir / "strain_response_225.png")
print(dashboard_dir / "strain_response_250.png")
print(output_csv)

print("\nCELL 9 STRAIN ANALYSIS COMPLETE")


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

project_dir = Path(
    "/content/drive/MyDrive/Project_4_Tidal_Turbine"
)

df = pd.read_csv(
    project_dir / "cleaned_data/test_250_complete.csv"
)

dashboard_dir = project_dir / "dashboards"
dashboard_dir.mkdir(parents=True, exist_ok=True)

# Investigate the suspected event interval.
event = df[
    df["elapsed_seconds"].between(60, 130)
].copy()

channels = [
    "Inboard Load",
    "Outboard Load",
    "Inboard Disp",
    "Outboard Disp",
    "SG37-2000-HP-SC",
    "SG38-2250-LP-SC",
    "SG39-2250-HP-SC"
]

fig, axes = plt.subplots(
    3, 1,
    figsize=(13, 11),
    sharex=True
)

# Structural loads
for col in channels[:2]:
    axes[0].plot(
        event["elapsed_seconds"],
        event[col],
        label=col
    )

axes[0].set_ylabel("Load (units unverified)")
axes[0].set_title("250% Test — Structural Event Investigation")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Displacements
for col in channels[2:4]:
    axes[1].plot(
        event["elapsed_seconds"],
        event[col],
        label=col
    )

axes[1].set_ylabel("Displacement (units unverified)")
axes[1].legend()
axes[1].grid(alpha=0.3)

# Strain-gauge signals
for col in channels[4:]:
    axes[2].plot(
        event["elapsed_seconds"],
        event[col],
        label=col
    )

axes[2].set_ylabel("Sensor signal (units unverified)")
axes[2].set_xlabel("Elapsed Time (seconds)")
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()

output = dashboard_dir / "structural_event_250.png"

plt.savefig(
    output,
    dpi=200,
    bbox_inches="tight"
)

plt.show()

# Identify the largest single-sample load decrease.
load_change = df["Inboard Load"].diff()
event_index = load_change.idxmin()

print("\nPROJECT 4: STRUCTURAL EVENT SUMMARY")
print("=" * 55)
print(
    "Largest inboard load drop at:",
    round(df.loc[event_index, "elapsed_seconds"], 3),
    "seconds"
)
print(
    "Single-sample load change:",
    round(load_change.loc[event_index], 3)
)
print("Event dashboard:", output)
print("\nCELL 10 EVENT ANALYSIS COMPLETE")


In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path

project_dir = Path(
    "/content/drive/MyDrive/Project_4_Tidal_Turbine"
)

df = pd.read_csv(
    project_dir / "cleaned_data/test_250_complete.csv"
)

event_time = 94.3
window = 1.0

channels = [
    "Inboard Load",
    "Outboard Load",
    "Inboard Disp",
    "Outboard Disp",
    "RootMomentUNCORRECTED",
    "SG37-2000-HP-SC",
    "SG38-2250-LP-SC",
    "SG39-2250-HP-SC"
]

# Compare one-second windows before and after event.
before = df[
    (df["elapsed_seconds"] >= event_time - window)
    & (df["elapsed_seconds"] < event_time)
]

after = df[
    (df["elapsed_seconds"] > event_time)
    & (df["elapsed_seconds"] <= event_time + window)
]

results = []

for channel in channels:
    before_mean = before[channel].mean()
    after_mean = after[channel].mean()

    results.append({
        "channel": channel,
        "before_mean": before_mean,
        "after_mean": after_mean,
        "absolute_change": after_mean - before_mean
    })

results_df = pd.DataFrame(results)

print("PROJECT 4: EVENT CORRELATION ANALYSIS")
print("=" * 60)
print("Event time:", event_time, "seconds")
print("Before samples:", len(before))
print("After samples:", len(after))
print("\nCHANNEL RESPONSE COMPARISON")
print(results_df.round(4).to_string(index=False))

output = (
    project_dir / "cleaned_data"
    / "structural_event_correlation.csv"
)

results_df.to_csv(output, index=False)

print("\nSaved:", output)
print("CELL 11 EVENT CORRELATION COMPLETE")


In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path

project_dir = Path(
    "/content/drive/MyDrive/Project_4_Tidal_Turbine"
)

df = pd.read_csv(
    project_dir / "cleaned_data/test_250_complete.csv"
)

event_time = 94.3

channels = [
    "elapsed_seconds",
    "Inboard Load",
    "Outboard Load",
    "Inboard Disp",
    "Outboard Disp",
    "RootMomentUNCORRECTED",
    "SG37-2000-HP-SC",
    "SG38-2250-LP-SC",
    "SG39-2250-HP-SC"
]

event_samples = df.loc[
    df["elapsed_seconds"].between(
        event_time - 0.5,
        event_time + 0.5
    ),
    channels
].copy()

print("PROJECT 4: STRUCTURAL EVENT VALIDATION")
print("=" * 60)

print("\nEVENT SAMPLE MEASUREMENTS")
print(
    event_samples.round(3).to_string(index=False)
)

sensors = [
    "SG37-2000-HP-SC",
    "SG38-2250-LP-SC",
    "SG39-2250-HP-SC"
]

before = df[
    df["elapsed_seconds"].between(90, 94.2)
]

after = df[
    df["elapsed_seconds"].between(95, 100)
]

print("\nSENSOR BEHAVIOR COMPARISON")

for sensor in sensors:
    print(f"\n{sensor}")
    print("Before mean:", round(before[sensor].mean(), 3))
    print("After mean:", round(after[sensor].mean(), 3))
    print("After std:", round(after[sensor].std(), 3))
    print("After min:", round(after[sensor].min(), 3))
    print("After max:", round(after[sensor].max(), 3))

output = (
    project_dir / "cleaned_data"
    / "structural_event_samples.csv"
)

event_samples.to_csv(output, index=False)

print("\nSaved:", output)
print("CELL 12 EVENT VALIDATION COMPLETE")


In [ ]:

import pandas as pd
import sqlite3
from pathlib import Path

project_dir = Path(
    "/content/drive/MyDrive/Project_4_Tidal_Turbine"
)

clean_dir = project_dir / "cleaned_data"
sql_dir = project_dir / "sql"
sql_dir.mkdir(parents=True, exist_ok=True)

db_path = sql_dir / "tidal_turbine_analytics.db"

df225 = pd.read_csv(
    clean_dir / "test_225_complete.csv"
)

df250 = pd.read_csv(
    clean_dir / "test_250_complete.csv"
)

conn = sqlite3.connect(db_path)

df225.to_sql(
    "test_225",
    conn,
    if_exists="replace",
    index=False
)

df250.to_sql(
    "test_250",
    conn,
    if_exists="replace",
    index=False
)

query = """
SELECT
    '225%' AS test_case,
    COUNT(*) AS records,
    MAX("Inboard Load") AS max_inboard_load,
    MAX("Outboard Load") AS max_outboard_load,
    MAX("RootMomentUNCORRECTED") AS max_root_moment
FROM test_225

UNION ALL

SELECT
    '250%' AS test_case,
    COUNT(*) AS records,
    MAX("Inboard Load"),
    MAX("Outboard Load"),
    MAX("RootMomentUNCORRECTED")
FROM test_250;
"""

results = pd.read_sql_query(query, conn)

print("PROJECT 4: SQL ENGINEERING ANALYSIS")
print("=" * 60)
print(results.round(3).to_string(index=False))

results.to_csv(
    sql_dir / "structural_test_sql_summary.csv",
    index=False
)

(sql_dir / "structural_test_summary.sql").write_text(
    query,
    encoding="utf-8"
)

conn.close()

print("\nDATABASE:", db_path)
print("SQL QUERY SAVED: structural_test_summary.sql")
print("CELL 13 SQL ANALYSIS COMPLETE")


In [ ]:

import sqlite3
import pandas as pd
from pathlib import Path

project_dir = Path(
    "/content/drive/MyDrive/Project_4_Tidal_Turbine"
)

sql_dir = project_dir / "sql"
db_path = sql_dir / "tidal_turbine_analytics.db"

query = """
WITH measurements AS (
    SELECT
        elapsed_seconds,
        "Inboard Load" AS inboard_load,
        "Outboard Load" AS outboard_load,
        "RootMomentUNCORRECTED" AS root_moment,

        LAG("Inboard Load") OVER (
            ORDER BY elapsed_seconds
        ) AS previous_inboard_load,

        LAG("Outboard Load") OVER (
            ORDER BY elapsed_seconds
        ) AS previous_outboard_load,

        LAG("RootMomentUNCORRECTED") OVER (
            ORDER BY elapsed_seconds
        ) AS previous_root_moment

    FROM test_250
),
changes AS (
    SELECT
        elapsed_seconds,
        inboard_load - previous_inboard_load
            AS inboard_change,
        outboard_load - previous_outboard_load
            AS outboard_change,
        root_moment - previous_root_moment
            AS root_moment_change
    FROM measurements
)
SELECT *
FROM changes
WHERE inboard_change IS NOT NULL
ORDER BY inboard_change ASC
LIMIT 10;
"""

with sqlite3.connect(db_path) as conn:
    results = pd.read_sql_query(query, conn)

print("PROJECT 4: ADVANCED SQL EVENT DETECTION")
print("=" * 60)
print(results.round(3).to_string(index=False))

results.to_csv(
    sql_dir / "sql_event_detection_results.csv",
    index=False
)

(sql_dir / "structural_event_detection.sql").write_text(
    query,
    encoding="utf-8"
)

print("\nSQL query saved.")
print("Event results CSV saved.")
print("CELL 14 ADVANCED SQL COMPLETE")


In [ ]:

from pathlib import Path
import pandas as pd

project_dir = Path(
    "/content/drive/MyDrive/Project_4_Tidal_Turbine"
)

report_dir = project_dir / "reports"
report_dir.mkdir(parents=True, exist_ok=True)

df225 = pd.read_csv(
    project_dir / "cleaned_data/test_225_complete.csv"
)

df250 = pd.read_csv(
    project_dir / "cleaned_data/test_250_complete.csv"
)

event = pd.read_csv(
    project_dir / "cleaned_data/structural_event_correlation.csv"
)

report = f"""# Marine Tidal Turbine Structural Performance Analytics

## Executive Summary

This project evaluates structural loading, displacement,
strain-gauge response, and test data quality for two
tidal turbine blade loading tests.

The analysis uses Python, Pandas, Matplotlib, and SQLite.

## Dataset Overview

Source: DOE TEAMER tidal turbine structural testing dataset

225% load case:
- Records: {len(df225):,}
- Maximum inboard load: {df225['Inboard Load'].max():.3f}
- Maximum outboard load: {df225['Outboard Load'].max():.3f}

250% load case:
- Records: {len(df250):,}
- Maximum inboard load: {df250['Inboard Load'].max():.3f}
- Maximum outboard load: {df250['Outboard Load'].max():.3f}

Total records: {len(df225) + len(df250):,}

## Data Quality

- No missing values in the loading-test tables.
- No duplicate measurement records.
- No infinite numeric values.
- Median sampling interval: 0.1 seconds.
- Three constant strain gauges in the 225% test.
- Four constant strain gauges in the 250% test.
- Constant gauges were flagged for review.

## Structural Event Findings

The 250% load-case test exhibited a significant
multi-channel event at approximately 94.3 seconds.

Single-sample changes:
- Inboard load: -3979.990
- Outboard load: -7347.755
- Uncorrected root moment: -85.629

The event was independently detected using Python
and SQLite window-function analysis.

## Strain-Gauge Findings

SG37, SG38, and SG39 transitioned to constant
negative readings following the event.

This behavior requires additional sensor-integrity
investigation before interpreting these readings
as valid physical strain.

## Event Window Comparison

{event.round(3).to_markdown(index=False)}

## Engineering Interpretation

The simultaneous changes in load, displacement,
root moment, and strain-gauge signals are consistent
with a major structural or test-system event.

Physical blade failure has not been independently
confirmed from the available analysis.

## Limitations

- Measurement units require technical documentation.
- The root moment channel is uncorrected.
- Constant sensor values may reflect instrumentation issues.
- The exact physical cause of the event remains unverified.
- The two tests have different recording durations.

## Analytical Tools

- Python
- Pandas
- NumPy
- Matplotlib
- SQLite
- Google Colab

## Portfolio Status

Engineering analysis completed through SQL event detection.
Final technical documentation and publication remain pending.
"""

report_path = (
    report_dir / "Project_4_Engineering_Findings.md"
)

report_path.write_text(
    report,
    encoding="utf-8"
)

print("PROJECT 4: ENGINEERING REPORT")
print("=" * 60)
print("Report saved:", report_path)
print("Report size:", report_path.stat().st_size, "bytes")
print("CELL 15 ENGINEERING REPORT COMPLETE")


In [ ]:

from pathlib import Path
import pandas as pd
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer,
    Table, TableStyle, Image, PageBreak
)
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.enums import TA_CENTER
from reportlab.lib.units import inch

project_dir = Path(
    "/content/drive/MyDrive/Project_4_Tidal_Turbine"
)

report_dir = project_dir / "reports"
dashboard_dir = project_dir / "dashboards"

pdf_path = report_dir / "Project_4_Engineering_Report.pdf"

styles = getSampleStyleSheet()
styles["Title"].alignment = TA_CENTER

doc = SimpleDocTemplate(
    str(pdf_path),
    pagesize=(612, 792),
    rightMargin=45,
    leftMargin=45,
    topMargin=45,
    bottomMargin=45
)

story = []

story.append(Paragraph(
    "MARINE TIDAL TURBINE",
    styles["Title"]
))
story.append(Paragraph(
    "Structural Performance & Engineering Analytics",
    styles["Heading2"]
))
story.append(Spacer(1, 20))

story.append(Paragraph(
    "Executive Summary", styles["Heading2"]
))
story.append(Paragraph(
    "This project analyzes tidal turbine blade structural "
    "testing using Python, Pandas, Matplotlib, and SQLite. "
    "The investigation compares 225% and 250% load-case "
    "tests and identifies a major multi-channel event "
    "at approximately 94.3 seconds during the 250% test.",
    styles["BodyText"]
))
story.append(Spacer(1, 15))

story.append(Paragraph(
    "Structural Test Comparison", styles["Heading2"]
))

data = [
    ["Measurement", "225% Test", "250% Test"],
    ["Records", "2,665", "9,630"],
    ["Maximum Inboard Load", "10,022.41", "11,174.15"],
    ["Maximum Outboard Load", "8,447.80", "8,023.32"],
    ["Maximum Root Moment", "127.33", "128.50"],
    ["Varying Strain Gauges", "26", "25"]
]

table = Table(data, colWidths=[215, 140, 140])
table.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#202B3A")),
    ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
    ("GRID", (0, 0), (-1, -1), 0.4, colors.grey),
    ("BOTTOMPADDING", (0, 0), (-1, -1), 9),
    ("TOPPADDING", (0, 0), (-1, -1), 9),
]))
story.append(table)
story.append(Spacer(1, 18))

story.append(Paragraph(
    "Structural Event Findings", styles["Heading2"]
))
story.append(Paragraph(
    "At 94.3 seconds, the 250% test recorded an inboard "
    "load change of -3,979.99, an outboard load change "
    "of -7,347.76, and an uncorrected root moment change "
    "of -85.629. These are single-sample differences. "
    "Measurement units remain unverified.",
    styles["BodyText"]
))
story.append(Spacer(1, 12))

story.append(Paragraph(
    "Sensor Integrity & Limitations", styles["Heading2"]
))
story.append(Paragraph(
    "Three strain-gauge channels, SG37, SG38, and SG39, "
    "transitioned to fixed negative readings near the "
    "event. This may indicate sensor or acquisition "
    "behavior and requires additional investigation. "
    "The physical cause of the event is not confirmed.",
    styles["BodyText"]
))

# Add existing engineering dashboard images.
images = [
    "structural_loading.png",
    "structural_displacement.png",
    "strain_response_225.png",
    "strain_response_250.png",
    "structural_event_250.png"
]

for filename in images:
    image_path = dashboard_dir / filename

    if image_path.exists():
        story.append(PageBreak())
        story.append(Paragraph(
            filename.replace("_", " ").replace(".png", "").title(),
            styles["Heading2"]
        ))
        story.append(Spacer(1, 12))
        story.append(Image(
            str(image_path),
            width=7.0 * inch,
            height=4.8 * inch,
            kind="proportional"
        ))

doc.build(story)

print("PROJECT 4: PDF ENGINEERING REPORT")
print("=" * 60)
print("PDF saved:", pdf_path)
print("File size:", round(
    pdf_path.stat().st_size / 1024, 2
), "KB")
print("CELL 16 PDF REPORT COMPLETE")


In [ ]:

%pip install reportlab


In [ ]:

import reportlab

print("ReportLab version:", reportlab.Version)
print("REPORTLAB INSTALLATION: PASS")


In [ ]:

%pip -q install pypdf

from pathlib import Path
from pypdf import PdfReader

project_dir = Path(
    "/content/drive/MyDrive/Project_4_Tidal_Turbine"
)

pdf_path = (
    project_dir / "reports"
    / "Project_4_Engineering_Report.pdf"
)

reader = PdfReader(str(pdf_path))

full_text = "\n".join(
    page.extract_text() or ""
    for page in reader.pages
)

required_sections = [
    "Executive Summary",
    "Structural Test Comparison",
    "Structural Event Findings",
    "Sensor Integrity",
]

expected_dashboards = [
    "structural_loading.png",
    "structural_displacement.png",
    "strain_response_225.png",
    "strain_response_250.png",
    "structural_event_250.png",
]

print("PROJECT 4: PDF VALIDATION")
print("=" * 60)

print("PDF exists:", pdf_path.exists())
print("Pages:", len(reader.pages))
print("File size KB:", round(
    pdf_path.stat().st_size / 1024, 2
))

print("\nREQUIRED REPORT SECTIONS")
sections_pass = True

for section in required_sections:
    found = section.lower() in full_text.lower()
    sections_pass &= found
    print(section, "PASS" if found else "MISSING")

print("\nDASHBOARD SOURCE FILES")
images_pass = True

for filename in expected_dashboards:
    exists = (
        project_dir / "dashboards" / filename
    ).exists()
    images_pass &= exists
    print(filename, "PASS" if exists else "MISSING")

print("\nEMBEDDED PDF IMAGES")
embedded_images = sum(
    len(page.images) for page in reader.pages
)
print("Image objects:", embedded_images)

pdf_pass = (
    len(reader.pages) >= 6
    and sections_pass
    and images_pass
    and embedded_images >= 5
)

print("\nPDF VALIDATION:",
      "PASS" if pdf_pass else "REVIEW REQUIRED")

print("CELL 17 PDF VALIDATION COMPLETE")


In [ ]:

from pathlib import Path

project_dir = Path(
    "/content/drive/MyDrive/Project_4_Tidal_Turbine"
)

readme_path = project_dir / "README.md"

readme = """# Marine Tidal Turbine Structural Performance Analytics

## Project Overview

Engineering data analytics portfolio project examining
tidal turbine structural loading, displacement,
strain-gauge response, and measurement quality.

The project compares 225% and 250% structural load-case
tests using Python and SQLite.

## Data Source

DOE TEAMER tidal turbine structural testing data.

The analysis includes:
- 225% load-case test: 2,665 records
- 250% load-case test: 9,630 records
- Total: 12,295 measurement records
- 29 strain-gauge channels per test

Original measurement units require verification
against source documentation.

## Engineering Objectives

1. Validate and prepare structural test measurements.
2. Compare loading and displacement response.
3. Evaluate strain-gauge variability.
4. Identify abrupt structural test events.
5. Investigate potential sensor-quality concerns.
6. Demonstrate Python and SQL engineering analytics.

## Key Findings

### Structural Test Comparison

| Metric | 225% Test | 250% Test |
|---|---:|---:|
| Records | 2,665 | 9,630 |
| Maximum inboard load | 10,022.414 | 11,174.147 |
| Maximum outboard load | 8,447.801 | 8,023.323 |
| Maximum uncorrected root moment | 127.332 | 128.497 |

### Significant Event: 250% Test

A major multi-channel event was detected at 94.3 seconds.

Single-sample changes:

| Channel | Change |
|---|---:|
| Inboard load | -3,979.990 |
| Outboard load | -7,347.755 |
| Uncorrected root moment | -85.629 |

Python time-series analysis and SQLite window
functions independently identified this event.

### Strain-Gauge Quality

Three channels (SG37, SG38, and SG39)
transitioned to constant negative readings
at the event.

This behavior requires further investigation
before assigning a physical failure mechanism.

## Engineering Interpretation

The 250% test contains a significant structural
or test-system event involving simultaneous
changes in loading, displacement, root moment,
and strain-gauge signals.

Physical blade failure has not been confirmed.

## Technologies

- Python
- Pandas
- NumPy
- Matplotlib
- SQLite
- Google Colab
- ReportLab

## Project Deliverables

- Cleaned structural measurement datasets
- Sensor quality summary
- Structural loading and displacement dashboards
- Strain-response dashboards
- Event investigation dashboard
- SQL database and queries
- Engineering findings Markdown report
- Six-page engineering PDF

## Project Limitations

- Measurement units have not been verified.
- Root moment measurements are uncorrected.
- Constant strain-gauge signals require review.
- Test durations differ.
- Event causation remains unverified.

## Reproducibility

The analysis was performed in Google Colab.
The final notebook, source-data documentation,
dependencies, and publication package will be
validated before the repository is declared complete.

## Status

Engineering analysis and automated PDF validation
completed. Final reproducibility and GitHub
publication checks pending.
"""

readme_path.write_text(
    readme,
    encoding="utf-8"
)

print("PROJECT 4: GITHUB README")
print("=" * 60)
print("README saved:", readme_path)
print("File size:", readme_path.stat().st_size, "bytes")

required = [
    "Project Overview",
    "Data Source",
    "Engineering Objectives",
    "Key Findings",
    "Engineering Interpretation",
    "Technologies",
    "Project Deliverables",
    "Project Limitations",
    "Reproducibility"
]

content = readme_path.read_text(encoding="utf-8")

for section in required:
    print(
        section,
        "PASS" if section in content else "MISSING"
    )

print("CELL 18 README COMPLETE")


In [ ]:

from pathlib import Path
import json
import sys
import importlib.metadata as metadata

project_dir = Path(
    "/content/drive/MyDrive/Project_4_Tidal_Turbine"
)

notebook_name = (
    "Project_4_Marine_Tidal_Turbine_Analytics.ipynb"
)

notebook_path = project_dir / notebook_name

print("PROJECT 4: NOTEBOOK & DEPENDENCY AUDIT")
print("=" * 60)

print("\nNOTEBOOK VERIFICATION")

if notebook_path.exists():

    with open(notebook_path, "r", encoding="utf-8") as f:
        notebook = json.load(f)

    cells = notebook.get("cells", [])

    code_cells = [
        cell for cell in cells
        if cell.get("cell_type") == "code"
    ]

    executable_cells = [
        cell for cell in code_cells
        if "".join(cell.get("source", [])).strip()
    ]

    print("Notebook:", notebook_name)
    print("Total cells:", len(cells))
    print("Code cells:", len(code_cells))
    print("Nonempty code cells:", len(executable_cells))
    print("Notebook file: PASS")

else:
    print("Notebook file: NOT FOUND")
    print("Expected:", notebook_path)
    print(
        "Save the current Colab notebook to the "
        "Project_4_Tidal_Turbine folder."
    )

print("\nDEPENDENCY VERSIONS")

packages = [
    "pandas",
    "numpy",
    "matplotlib",
    "reportlab",
    "pypdf",
    "openpyxl",
    "tabulate"
]

requirements = []

for package in packages:
    try:
        version = metadata.version(package)
        requirements.append(f"{package}=={version}")
        print(f"{package}: {version}")
    except metadata.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

requirements_path = project_dir / "requirements.txt"

requirements_path.write_text(
    "\n".join(requirements) + "\n",
    encoding="utf-8"
)

print("\nPython:", sys.version.split()[0])
print("Requirements saved:", requirements_path)

print("\nFINAL AUDIT STATUS")

if notebook_path.exists() and len(executable_cells) >= 15:
    print("NOTEBOOK STRUCTURE: PASS")
else:
    print("NOTEBOOK STRUCTURE: REVIEW REQUIRED")

print("CELL 19 NOTEBOOK AUDIT COMPLETE")


In [ ]:

from pathlib import Path
import json

project_dir = Path(
    "/content/drive/MyDrive/Project_4_Tidal_Turbine"
)

notebook_path = (
    project_dir /
    "Project_4_Marine_Tidal_Turbine_Analytics.ipynb"
)

print("PROJECT 4: NOTEBOOK RECOVERY")
print("=" * 60)

if notebook_path.exists():
    with open(notebook_path, "r", encoding="utf-8") as f:
        notebook = json.load(f)

    cells = notebook.get("cells", [])
    code_cells = [
        c for c in cells
        if c.get("cell_type") == "code"
    ]

    print("Notebook found: PASS")
    print("Total cells:", len(cells))
    print("Code cells:", len(code_cells))
    print("File size KB:",
          round(notebook_path.stat().st_size / 1024, 2))
else:
    print("Notebook found: NOT YET")
    print("\nSearching Google Drive...")

    matches = list(
        Path("/content/drive/MyDrive").rglob(
            "*Tidal*Turbine*.ipynb"
        )
    )

    for match in matches[:20]:
        print("FOUND:", match)

    if not matches:
        print("No matching notebook found.")

print("\nCELL 19A NOTEBOOK RECOVERY CHECK COMPLETE")
